In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
from scipy.interpolate import RegularGridInterpolator


In [4]:
fname = f'/srv/scratch/z3533156/26year_BRAN2020/outer_avg_01461.nc'
dataset = nc.Dataset(fname)
lon_rho = np.transpose(dataset.variables['lon_rho'], axes=(1, 0))
lat_rho = np.transpose(dataset.variables['lat_rho'], axes=(1, 0))
mask_rho = np.transpose(dataset.variables['mask_rho'], axes=(1, 0))
angle = dataset.variables['angle'][0, 0]
def distance(lat1, lon1, lat2, lon2):
    EARTH_RADIUS = 6357
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return EARTH_RADIUS * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
j_mid, i_mid = lon_rho.shape[1] // 2, lon_rho.shape[0] // 2
dx = distance(lat_rho[:-1, j_mid], lon_rho[:-1, j_mid],
              lat_rho[1:, j_mid], lon_rho[1:, j_mid])
dy = distance(lat_rho[i_mid, :-1], lon_rho[i_mid, :-1],
              lat_rho[i_mid, 1:], lon_rho[i_mid, 1:])
x_grid = np.insert(np.cumsum(dx), 0, 0)
y_grid = np.insert(np.cumsum(dy), 0, 0)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid, indexing='ij')


In [18]:
df_eddies_untracked = pd.read_pickle('df_eddies_untracked_orig.pkl')
df_eddies_untracked['Cyc']  = df_eddies_untracked['nCyc']
# Find Lon and Lat values
xg, yg = x_grid, y_grid
lon_interp = RegularGridInterpolator((yg, xg), lon_rho.T, bounds_error=False, fill_value=np.nan)
lat_interp = RegularGridInterpolator((yg, xg), lat_rho.T, bounds_error=False, fill_value=np.nan)
points = np.column_stack((df_eddies_untracked['yc'].to_numpy(), df_eddies_untracked['xc'].to_numpy()))
df_eddies_untracked['lon'] = lon_interp(points)
df_eddies_untracked['lat'] = lat_interp(points)
df_eddies_untracked['fname'] = [
    f"/srv/scratch/z3533156/26year_BRAN2020/outer_avg_{1461 + ((day - 1462) // 30) * 30:05}.nc"
    for day in df_eddies_untracked['Day']
]
df_eddies_untracked = df_eddies_untracked.drop(columns=['nxc', 'nyc', 'nCyc', 'fnumber', 'Omega0', 'R'])
df_eddies_untracked['Eddy'] = df_eddies_untracked.groupby('Day').cumcount() + 1
df_eddies_untracked = df_eddies_untracked[['Day', 'Eddy', 'Cyc', 'xc', 'yc', 'lon','lat', 'w', 'Q', 'Omega', 'Rc', 'psi0', 'fname']]
df_eddies_untracked.to_pickle('df_eddies_untracked.pkl')
df_eddies_untracked


,Day,Eddy,Cyc,xc,yc,lon,lat,w,Q,Omega,Rc,psi0,fname
0,1462,1,AE,829.879126,1515.297884,160.568762,-28.072350,0.015631,"[[0.7054724746035225, -0.17299518294950286], [...",0.008268,96.456316,-38.462785,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
1,1462,2,AE,357.902634,1407.794571,155.723491,-27.536492,0.031539,"[[1.2525415454561337, -0.37318304544145586], [...",0.013549,78.042405,-41.260426,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
2,1462,3,CE,928.181752,1356.070550,161.072036,-29.736582,-0.010848,"[[0.9184334895938384, -0.6568144522261018], [-...",-0.006890,118.032741,47.992205,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
3,1462,4,CE,505.750855,1353.499933,156.969877,-28.456477,-0.032406,"[[1.0915204469734063, -0.17483272531676586], [...",-0.013202,106.666574,75.106895,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
4,1462,5,AE,753.849693,1284.689807,159.168134,-29.809701,0.022045,"[[1.300245004042572, -0.37537759847059776], [-...",0.008255,103.327517,-44.066805,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
416246,10650,25,AE,348.663439,158.138527,151.457193,-38.216042,0.033157,"[[1.3863808771116255, 0.04649055064735413], [0...",0.011945,110.552001,-72.997132,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
416247,10650,26,CE,978.294053,127.970950,158.258343,-40.415935,-0.003838,"[[0.6233322936879349, -0.5502050858849237], [-...",-0.001503,75.127332,4.241051,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
416248,10650,27,AE,804.765913,95.382012,156.241400,-40.161029,0.010840,"[[1.2619918374118362, 0.3980497594289241], [0....",0.005905,60.426625,-10.779905,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
416249,10650,28,CE,157.210669,33.449590,148.933721,-38.697131,-0.007272,"[[0.9874338034387458, 0.628992620695803], [0.6...",-0.003622,141.142981,36.082072,/srv/scratch/z3533156/26year_BRAN2020/outer_av...


In [13]:
df_eddies_untracked.columns

Index(['Day', 'xc', 'yc', 'w', 'Q', 'Omega', 'Rc', 'psi0', 'R', 'Cyc', 'lon',
       'lat', 'fname'],
      dtype='object')